In [1]:
import pyspark
import time
from pyspark import SparkContext

In [2]:
sc = SparkContext(master='local[2]')

In [3]:
sc

<SparkContext master=local[2] appName=pyspark-shell>

In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Logistic Regression").getOrCreate()

In [5]:
start_time1 = time.time()
true_df = spark.read.csv("True.csv", inferSchema=True, header=True)
fake_df = spark.read.csv("Fake.csv", inferSchema=True, header=True)
end_time1 = time.time()

In [6]:
true_df.show(5)

+--------------------+--------------------+------------+------------------+
|               title|                text|     subject|              date|
+--------------------+--------------------+------------+------------------+
|As U.S. budget fi...|WASHINGTON (Reute...|politicsNews|December 31, 2017 |
|U.S. military to ...|WASHINGTON (Reute...|politicsNews|December 29, 2017 |
|Senior U.S. Repub...|WASHINGTON (Reute...|politicsNews|December 31, 2017 |
|FBI Russia probe ...|WASHINGTON (Reute...|politicsNews|December 30, 2017 |
|Trump wants Posta...|SEATTLE/WASHINGTO...|politicsNews|December 29, 2017 |
+--------------------+--------------------+------------+------------------+
only showing top 5 rows



In [7]:
fake_df.show(5)

+--------------------+--------------------+-------+-----------------+
|               title|                text|subject|             date|
+--------------------+--------------------+-------+-----------------+
| Donald Trump Sen...|Donald Trump just...|   News|December 31, 2017|
| Drunk Bragging T...|House Intelligenc...|   News|December 31, 2017|
| Sheriff David Cl...|On Friday, it was...|   News|December 30, 2017|
| Trump Is So Obse...|On Christmas day,...|   News|December 29, 2017|
| Pope Francis Jus...|Pope Francis used...|   News|December 25, 2017|
+--------------------+--------------------+-------+-----------------+
only showing top 5 rows



In [8]:
from pyspark.sql.functions import lit, rand

In [9]:
start_time2 = time.time()
data= true_df.withColumn('trueOrfake', lit(1)).union(fake_df.withColumn('trueOrfake', lit(0))).orderBy(rand())
end_time2 = time.time()

In [10]:
data = data.select(["text", "subject", 'trueOrfake'])

In [11]:
desired_values = ["politicsNews", "News", "politics", "Government News", "left-news", "US_News", "Middle-east", "worldnews"]

In [12]:
from pyspark.sql.functions import col

In [13]:
start_time3 = time.time()
df_filtered = data.filter(col("subject").isin(desired_values))
end_time3 = time.time()

In [14]:
df_filtered.groupby('subject').count().sort("count",ascending=False).show()

+---------------+-----+
|        subject|count|
+---------------+-----+
|   politicsNews|11209|
|      worldnews|10115|
|           News| 8501|
|       politics| 6525|
|      left-news| 4216|
|Government News| 1543|
|        US_News|  767|
|    Middle-east|  762|
+---------------+-----+



In [15]:
df_filtered.toPandas()['text'].isnull().sum()

0

In [16]:
df_filtered.toPandas()['subject'].isnull().sum()

0

In [17]:
df_filtered.show(5)

+--------------------+------------+----------+
|                text|     subject|trueOrfake|
+--------------------+------------+----------+
|The rampant migra...|   left-news|         0|
|MEXICO CITY (Reut...|   worldnews|         1|
|21st Century Wire...|     US_News|         0|
|CHICAGO (Reuters)...|politicsNews|         1|
|MOSCOW (Reuters) ...|politicsNews|         1|
+--------------------+------------+----------+
only showing top 5 rows



In [18]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF, StringIndexer

In [19]:
# Stages for the pipeline
start_time4 = time.time()
tokenizer = Tokenizer(inputCol='text', outputCol='mytokens')
stopwordRemover = StopWordsRemover(inputCol='mytokens',outputCol='filtered_tokens')
vectorizer = CountVectorizer(inputCol='filtered_tokens',outputCol='rawFeatures')
idf = IDF(inputCol='rawFeatures', outputCol='vectorizedFeatures')
labelEncoder = StringIndexer(inputCol='subject',outputCol='label').fit(df_filtered)
end_time4 = time.time()

In [20]:
labelEncoder.transform(df_filtered).show(5)

+--------------------+------------+----------+-----+
|                text|     subject|trueOrfake|label|
+--------------------+------------+----------+-----+
|The rampant migra...|   left-news|         0|  4.0|
|MEXICO CITY (Reut...|   worldnews|         1|  1.0|
|21st Century Wire...|     US_News|         0|  6.0|
|CHICAGO (Reuters)...|politicsNews|         1|  0.0|
|MOSCOW (Reuters) ...|politicsNews|         1|  0.0|
+--------------------+------------+----------+-----+
only showing top 5 rows



In [21]:
start_time5 = time.time()
df_filtered = labelEncoder.transform(df_filtered)
end_time5 = time.time()

In [22]:
(train_df, test_df) = df_filtered.randomSplit((0.7,0.3),seed=42)

In [23]:
train_df.count()

30733

In [24]:
test_df.count()

12905

In [25]:
train_df.show(2)

+----+---------------+----------+-----+
|text|        subject|trueOrfake|label|
+----+---------------+----------+-----+
|    |Government News|         0|  5.0|
|    |Government News|         0|  5.0|
+----+---------------+----------+-----+
only showing top 2 rows



In [27]:
# machine learning model (Estimator) (data to model)
from pyspark.ml.classification import NaiveBayes
nb = NaiveBayes(smoothing=1.0, modelType="multinomial", featuresCol='vectorizedFeatures',
                        labelCol = 'trueOrfake')

In [28]:
from pyspark.ml import Pipeline

In [29]:
pipeline = Pipeline(
    stages=[tokenizer, stopwordRemover, vectorizer, idf, nb]
)

In [30]:
pipeline.stages

Param(parent='Pipeline_18f1971e05d8', name='stages', doc='a list of pipeline stages')

In [31]:
start_time6 = time.time()
nb_model = pipeline.fit(train_df)
end_time6 = time.time()

In [32]:
nb_model

PipelineModel_997df7c3702f

In [33]:
#testare
start_time7 = time.time()
predictions = nb_model.transform(test_df)
end_time7 = time.time()

In [34]:
predictions.columns

['text',
 'subject',
 'trueOrfake',
 'label',
 'mytokens',
 'filtered_tokens',
 'rawFeatures',
 'vectorizedFeatures',
 'rawPrediction',
 'probability',
 'prediction']

In [36]:
predictions.select('vectorizedFeatures', 'rawPrediction', 'probability','subject','label','trueOrFake','prediction').show(5000)

+--------------------+--------------------+--------------------+---------------+-----+----------+----------+
|  vectorizedFeatures|       rawPrediction|         probability|        subject|label|trueOrFake|prediction|
+--------------------+--------------------+--------------------+---------------+-----+----------+----------+
|      (262144,[],[])|[-0.6660885753192...|[0.51371400683260...|Government News|  5.0|         0|       0.0|
|      (262144,[],[])|[-0.6660885753192...|[0.51371400683260...|Government News|  5.0|         0|       0.0|
|      (262144,[],[])|[-0.6660885753192...|[0.51371400683260...|Government News|  5.0|         0|       0.0|
|      (262144,[],[])|[-0.6660885753192...|[0.51371400683260...|Government News|  5.0|         0|       0.0|
|      (262144,[],[])|[-0.6660885753192...|[0.51371400683260...|Government News|  5.0|         0|       0.0|
|      (262144,[],[])|[-0.6660885753192...|[0.51371400683260...|Government News|  5.0|         0|       0.0|
|      (262144,[],[

In [37]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

In [38]:
evaluator = MulticlassClassificationEvaluator(predictionCol='prediction',labelCol='trueOrfake')

In [39]:
accuracy = evaluator.evaluate(predictions)
accuracy*100

98.72146643407653

In [40]:
bin_eval = BinaryClassificationEvaluator(rawPredictionCol='prediction', labelCol='trueOrfake')

In [41]:
roc = bin_eval.evaluate(predictions)
print(roc*100)

98.72682146352787


In [42]:
auPR = bin_eval.evaluate(predictions, {bin_eval.metricName: "areaUnderPR"})
print(auPR*100)

98.03528600291963


In [43]:
evaluator.setMetricName('truePositiveRateByLabel')
evaluator.setMetricLabel(1.0)
print(evaluator.evaluate(predictions))

0.9920050164602602


In [44]:
evaluator.setMetricName('truePositiveRateByLabel')
evaluator.setMetricLabel(0.0)
print(evaluator.evaluate(predictions))

0.9825314128102973


In [45]:
evaluator.setMetricName('falsePositiveRateByLabel')
evaluator.setMetricLabel(1.0)
print(evaluator.evaluate(predictions))

0.017468587189702726


In [46]:
evaluator.setMetricName('falsePositiveRateByLabel')
evaluator.setMetricLabel(0.0)
print(evaluator.evaluate(predictions))

0.007994983539739771


In [47]:
execution_time1 = end_time1 - start_time1 #timp de executie pentru citirea datelor
execution_time2 = end_time2 - start_time2 #timp de executie pentru combinarea datelor si amestecarea lor (preprocesarea datelor)
execution_time3 = end_time3 - start_time3 #timp de executie pentru pastrarea valorilor corecte pentru coloana subject (preprocesarea datelor)
execution_time4 = end_time4 - start_time4 #timp de executie pentru procesarea datelor
execution_time5 = end_time5 - start_time5 #timp de executie pentru procesarea datelor 

execution_time6 = end_time6 - start_time6 #timp de executie pentru antrenarea modelului
execution_time7 = end_time7 - start_time7 #timp de executie pentru testare

total_execution_time_for_preprocessing_data = execution_time2 + execution_time3
total_execution_time_for_processing_data = execution_time4 + execution_time5
total_execution_time_for_testing = execution_time7 

print("Timp de executie pentru citirea datelor: {} secunde".format(execution_time1))
print("Timp de executie pentru preprocesarea datelor: {} secunde".format(total_execution_time_for_preprocessing_data))
print("Timp de executie pentru procesarea datelor: {} secunde".format(total_execution_time_for_processing_data))
print("Timp de executie pentru antrenarea modelului: {} secunde".format(execution_time6))
print("Timp de executie pentru testare: {} secunde".format(total_execution_time_for_testing))

Timp de executie pentru citirea datelor: 7.162604331970215 secunde
Timp de executie pentru preprocesarea datelor: 0.10168242454528809 secunde
Timp de executie pentru procesarea datelor: 2.1849722862243652 secunde
Timp de executie pentru antrenarea modelului: 34.1334023475647 secunde
Timp de executie pentru testare: 0.2040843963623047 secunde
